# synthpose-webots — Geração de dataset COCO-pose do NAO no Kaggle

Este notebook roda o pipeline de geração de dataset sintético (Blender 5.1.2 headless + Cycles GPU)
diretamente em um **Kaggle Notebook**.

## Antes de rodar (menu à direita do Kaggle)
1. **Settings → Accelerator → GPU** (T4 x2 ou P100). Necessário para o Cycles renderizar rápido.
2. **Settings → Internet → On**. Necessário para baixar o Blender e clonar o repositório.

## O que o notebook faz
1. Baixa e extrai o **Blender 5.1.2** (Linux x64).
2. Clona o repositório `synthpose-webots`.
3. Instala `pyyaml` no Python embutido do Blender (numpy já vem embutido; `cv2`/`pycocotools`
   só são usados na validação, que roda no Python do Kaggle).
4. Habilita a **GPU no Cycles** (OptiX → CUDA) via um wrapper.
5. Roda `dataset_generator_phobos.py` gravando em `/kaggle/working/output`.
6. Valida o JSON COCO e mostra uma amostra renderizada.
7. Compacta a saída para download.


## 1. Configuração

Defina o **job** na célula abaixo: `TOTAL_SAMPLES` (total do dataset inteiro),
`WORLD_SIZE` (quantas máquinas Kaggle) e `RANK` (índice desta máquina, de 0).

Cada máquina pega uma **fatia disjunta** (`shard_range`) e, dentro dela, divide o
trabalho entre as **GPUs locais** — 1 processo Blender por T4 (a T4 x2 do Kaggle
gera em paralelo). Nomes de arquivo e sementes nunca colidem entre máquinas nem
entre GPUs, então o merge no fim só renumera os IDs COCO.

**2 máquinas:** mesmo `TOTAL_SAMPLES`/`WORLD_SIZE=2`, muda só `RANK` (0 e 1).
**1 máquina de 2 T4:** deixe `WORLD_SIZE=1, RANK=0` — as duas GPUs já são usadas.
O Kaggle limita a sessão a ~9h; comece com `TOTAL_SAMPLES` pequeno (ex.: 40) para
validar antes do lote grande.

> **Repositório privado?** Se `git clone` falhar por autenticação, faça upload do repo como um
> *Kaggle Dataset* e aponte `REPO_DIR` para `/kaggle/input/<seu-dataset>/synthpose-webots`
> (pule a célula de clone).

In [ ]:
import os

# --- Blender ---
BLENDER_VERSION = "5.1.2"
BLENDER_SERIES  = "Blender5.1"
BLENDER_URL = f"https://download.blender.org/release/{BLENDER_SERIES}/blender-{BLENDER_VERSION}-linux-x64.tar.xz"

# --- Repositorio ---
REPO_URL    = "https://github.com/Rinaldots/synthpose-webots.git"
REPO_BRANCH = "main"
REPO_DIR    = "/kaggle/working/synthpose-webots"   # ajuste p/ /kaggle/input/... se usar Dataset

# --- Job (quantas amostras no TOTAL, somando todas as maquinas) ---
TOTAL_SAMPLES = 10000

# --- Coordenacao entre maquinas ---
# DYNAMIC=True (recomendado): AUTOMATICO, dispensa RANK. Cada maquina reivindica o
#   proximo LOTE (chunk) livre no armazenamento compartilhado, continua do que ja
#   existe e nunca sobrepoe outra. Ligue/desligue maquinas a vontade. EXIGE backup
#   (o remote e a coordenacao + o armazenamento). Nao precisa mexer em WORLD_SIZE/RANK.
# DYNAMIC=False: modo manual — voce define RANK por maquina (fatia estatica).
DYNAMIC    = True
CHUNK      = 500     # tamanho do lote que cada maquina reivindica por vez (so DYNAMIC)
WORLD_SIZE = 2       # (so DYNAMIC=False) quantas maquinas Kaggle vao rodar
RANK       = 0       # (so DYNAMIC=False) <<< MUDE POR MAQUINA: 0, 1, ...

OUT_DIR = "/kaggle/working/output"    # disco local temporario (backup libera o espaco)
USE_GPU = True

# --- Desempenho ---
# PROCS_PER_GPU: quantos processos Blender por GPU. Como a GPU fica ociosa durante
# o setup de cena (CPU), rodar 2 por GPU sobrepoe o setup de um com o render de
# outro -> mais throughput. 2 costuma ser o ponto doce nos ~4 vCPU do Kaggle.
PROCS_PER_GPU = 2
# SAVE_EVERY: grava o JSON COCO a cada N imagens (checkpoint). =1 reescreve o JSON
# INTEIRO a cada frame (fica O(N^2) conforme cresce). 100 mantem retomada barata.
SAVE_EVERY = 100

# --- Monitoramento remoto (opcional, via ntfy.sh) ---
# Deixe "" para desativar. Com um topico setado, cada maquina PUBLICA seu status
# (GPU + progresso) e voce agrega tudo no seu PC com:
#   python dataset_generator/scripts/monitor.py --subscribe <topico> --world-size 2
# Use um nome LONGO e ALEATORIO: funciona como senha (quem souber o topico, le).
NTFY_TOPIC = "synthpose-rinaldo-9f3a2b7c1d"   # ex.: "synthpose-rinaldo-9f3a2b7c1d"

# --- Onde armazenar (backup incremental + coordenacao dos lotes) ---
# O monitor MOVE as imagens pro remote (apaga a copia local -> o disco do Kaggle nao
# enche) e mantem os JSON locais. Escolha o backend:
#   "drive"     -> Google Drive via rclone.conf (cole o bloco [gdrive] em RCLONE_CONF).
#                  Simples, mas tem cota de API (erro 403 'Queries per minute') e so
#                  15GB gratis. Se usar, crie seu proprio client_id (evita o 403).
#   "tailscale" -> SEU PC via Tailscale + `rclone serve webdav` (espaco do seu disco,
#                  sem cota). Preencha TS_*/PC_* na celula "Storage: Tailscale" e veja
#                  dataset_generator/docs/storage_tailscale.md. Exige o PC LIGADO.
STORAGE      = "tailscale"     # "drive" ou "tailscale"
BACKUP_EVERY = 100
if STORAGE == "drive":
    BACKUP_REMOTE = ""         # ex.: "gdrive:synthpose-backup"
    # Cole aqui o bloco do rclone.conf (com [gdrive], type=drive, token=..., e de
    # preferencia client_id/client_secret PROPRIOS p/ nao esbarrar na cota).
    RCLONE_CONF = ""
else:                          # tailscale: o remote 'pc' e definido na celula de setup
    BACKUP_REMOTE = "pc:kaggle"
    RCLONE_CONF = ""

WORK = "/kaggle/working"
os.makedirs(WORK, exist_ok=True)
assert DYNAMIC or (0 <= RANK < WORLD_SIZE), "RANK deve estar em [0, WORLD_SIZE)"
assert (not DYNAMIC) or BACKUP_REMOTE, \
    "DYNAMIC=True exige um remote de backup (o remote coordena + armazena)."
_mode = f"DINAMICO chunk={CHUNK}" if DYNAMIC else f"MANUAL world_size={WORLD_SIZE} rank={RANK}"
print(f"Config OK  |  total={TOTAL_SAMPLES}  {_mode}  storage={STORAGE}"
      f"  procs/gpu={PROCS_PER_GPU}  save_every={SAVE_EVERY}"
      + (f"  ntfy={NTFY_TOPIC}" if NTFY_TOPIC else "  (ntfy off)")
      + (f"  backup->{BACKUP_REMOTE}(/{BACKUP_EVERY})" if BACKUP_REMOTE else "  (backup off)"))

In [ ]:
# --- Setup do rclone p/ backup no DRIVE (roda so se STORAGE=="drive") ---
# Como configurar UMA vez, no SEU PC:
#   1) instale o rclone e rode `rclone config`
#   2) crie um remote do tipo Google Drive (faz o login no navegador)
#   3) abra ~/.config/rclone/rclone.conf, copie o bloco INTEIRO, COMECANDO pela
#      linha do cabecalho entre colchetes, ex.: [123]
#   4) cole em RCLONE_CONF (celula anterior) e ponha BACKUP_REMOTE="123:synthpose-backup"
# DICA: crie seu proprio client_id (console.cloud.google.com) e inclua client_id/
#   client_secret no bloco — evita o erro 403 'Quota exceeded / Queries per minute'
#   (a cota compartilhada padrao do rclone estoura). Se preferir o seu PC, use
#   STORAGE="tailscale" (proxima celula) e ignore esta.
import os, shutil, subprocess

RCLONE_CONFIG_PATH = "/kaggle/working/rclone.conf"

def _ensure_rclone():
    if shutil.which("rclone"):
        return True
    print("Instalando rclone via apt...")
    subprocess.run("apt-get -qq update && apt-get -qq install -y rclone 2>/dev/null", shell=True)
    if shutil.which("rclone"):
        return True
    print("apt falhou; tentando script oficial...")
    subprocess.run("curl -s https://rclone.org/install.sh | sudo bash", shell=True)
    return bool(shutil.which("rclone"))

if STORAGE == "drive" and BACKUP_REMOTE:
    assert _ensure_rclone(), "rclone nao pode ser instalado; backup indisponivel"
    remote_name = BACKUP_REMOTE.split(":")[0]

    if RCLONE_CONF.strip():
        if f"[{remote_name}]" not in RCLONE_CONF:
            raise ValueError(
                f"RCLONE_CONF nao contem a secao [{remote_name}]. Cole o bloco INTEIRO "
                f"do rclone.conf, comecando pela linha [{remote_name}] (o cabecalho entre "
                f"colchetes). Ex.: '[{remote_name}]\\ntype = drive\\n...'")
        with open(RCLONE_CONFIG_PATH, "w") as f:
            f.write(RCLONE_CONF)
        # Aponta TODAS as chamadas de rclone (inclusive o backup do monitor, que roda
        # neste mesmo processo) para este arquivo — evita ambiguidade de caminho.
        os.environ["RCLONE_CONFIG"] = RCLONE_CONFIG_PATH
        print("rclone.conf escrito em", RCLONE_CONFIG_PATH)

    print(subprocess.run(["rclone", "version"], capture_output=True, text=True).stdout.splitlines()[0])
    lr = subprocess.run(["rclone", "listremotes"], capture_output=True, text=True)
    print("remotes carregados:", lr.stdout.replace("\n", " ").strip() or "(NENHUM!)")
    assert f"{remote_name}:" in lr.stdout, \
        f"remote '{remote_name}:' nao carregado — confira o cabecalho [{remote_name}] no RCLONE_CONF"
    # Teste de conexao. Com scope=drive.file, 'lsd' na raiz pode vir vazio (rc=0) — ok:
    # o rclone so enxerga o que ele mesmo cria; o 'copy' cria a pasta na 1a vez.
    r = subprocess.run(["rclone", "lsd", remote_name + ":"], capture_output=True, text=True)
    print("rclone remote OK" if r.returncode == 0 else f"conexao FALHOU:\n{r.stderr[:400]}")
elif STORAGE == "tailscale":
    print("STORAGE=tailscale: configure na proxima celula ('Storage: Tailscale').")
else:
    print("Backup desativado (BACKUP_REMOTE vazio).")

In [ ]:
# --- Storage: Tailscale -> SEU PC (roda so se STORAGE=="tailscale") ---
# Pre-requisitos NO SEU PC (uma vez) — ver dataset_generator/docs/storage_tailscale.md:
#   curl -fsSL https://tailscale.com/install.sh | sh && sudo tailscale up
#   tailscale ip -4                                   # -> PC_TSIP
#   rclone serve webdav ~/synthpose-backup --addr 0.0.0.0:8080 --user U --pass P
#   (deixe rodando durante toda a geracao; tmux/screen)
import os, subprocess, shutil, time

# >>> PREENCHA <<<
TS_AUTHKEY     = ""            # chave efemera/reutilizavel do admin console do Tailscale
PC_TSIP        = "100.x.y.z"   # IP do `tailscale ip -4` no seu PC
PC_WEBDAV_PORT = 8080
PC_WEBDAV_USER = "rinaldo"     # o mesmo --user do `rclone serve webdav`
PC_WEBDAV_PASS = ""           # a mesma --pass do `rclone serve webdav`

if STORAGE == "tailscale":
    assert TS_AUTHKEY and PC_WEBDAV_PASS and PC_TSIP != "100.x.y.z", \
        "Preencha TS_AUTHKEY, PC_TSIP e PC_WEBDAV_PASS."
    os.environ["PATH"] += ":/usr/sbin:/usr/bin"     # tailscaled costuma ficar em /usr/sbin
    # 1. Instala tailscale + rclone recente (o apt traz um rclone velho, 1.53).
    if not shutil.which("tailscaled"):
        subprocess.run("curl -fsSL https://tailscale.com/install.sh | sh", shell=True)
    if not shutil.which("rclone"):
        subprocess.run("curl -fsSL https://rclone.org/install.sh | sudo bash", shell=True)
    # 2. tailscaled em modo USERSPACE (Kaggle nao expoe /dev/net/tun) + proxy HTTP de
    #    saida na 1055 — e por ele que o rclone alcanca o PC pela tailnet.
    subprocess.Popen(["tailscaled", "--tun=userspace-networking", "--state=mem:",
                      "--outbound-http-proxy-listen=localhost:1055"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(3)
    up = subprocess.run(["tailscale", "up", "--authkey", TS_AUTHKEY,
                         "--hostname", "kaggle-synthpose"], capture_output=True, text=True)
    print("tailscale up:", "ok" if up.returncode == 0 else up.stderr[:300])
    # 3. Roteia as chamadas HTTP do rclone pelo proxy do tailscale (herdado pelos
    #    subprocessos de backup/coordenacao, que rodam sem env explicito).
    os.environ["HTTP_PROXY"] = os.environ["HTTPS_PROXY"] = "http://localhost:1055"
    os.environ["NO_PROXY"] = "localhost,127.0.0.1"
    # 4. Define o remote 'pc' (webdav) via env — sem tocar em rclone.conf.
    obsc = subprocess.run(["rclone", "obscure", PC_WEBDAV_PASS],
                          capture_output=True, text=True).stdout.strip()
    os.environ["RCLONE_CONFIG_PC_TYPE"]   = "webdav"
    os.environ["RCLONE_CONFIG_PC_URL"]    = f"http://{PC_TSIP}:{PC_WEBDAV_PORT}"
    os.environ["RCLONE_CONFIG_PC_VENDOR"] = "rclone"
    os.environ["RCLONE_CONFIG_PC_USER"]   = PC_WEBDAV_USER
    os.environ["RCLONE_CONFIG_PC_PASS"]   = obsc
    # 5. Valida a conexao ate o PC (lista a raiz servida no WebDAV).
    r = subprocess.run(["rclone", "lsd", "pc:"], capture_output=True, text=True)
    print("conexao PC OK" if r.returncode == 0
          else f"conexao FALHOU (cheque o serve webdav / PC_TSIP / senha):\n{r.stderr[:400]}")
else:
    print("STORAGE != tailscale; pulando setup do Tailscale.")

## 2. Baixar e extrair o Blender 5.1.2

In [ ]:
import os, stat, shutil, subprocess, glob, tarfile, urllib.request

TARBALL = f"{WORK}/blender.tar.xz"
BLENDER_HOME = f"{WORK}/blender-{BLENDER_VERSION}-linux-x64"

if not os.path.isdir(BLENDER_HOME):
    # download.blender.org recusa User-Agent nao-navegador (HTTP 403);
    # por isso enviamos um User-Agent de navegador.
    if not os.path.exists(TARBALL) or os.path.getsize(TARBALL) < 1_000_000:
        print("Baixando Blender...", BLENDER_URL)
        req = urllib.request.Request(BLENDER_URL, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as r, open(TARBALL, "wb") as f:
            shutil.copyfileobj(r, f)
    print("Extraindo...")
    with tarfile.open(TARBALL) as t:
        t.extractall(WORK)

BLENDER = os.path.join(BLENDER_HOME, "blender")
assert os.path.exists(BLENDER), f"blender nao encontrado em {BLENDER}"

# Restaura o bit de execucao: a persistencia do /kaggle/working entre sessoes pode
# remover +x do binario do Blender e do Python embutido (causa PermissionError).
for exe in [BLENDER] + glob.glob(f"{BLENDER_HOME}/*/python/bin/python3*"):
    os.chmod(exe, os.stat(exe).st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

# Bibliotecas de sistema que o Blender 5.1 headless exige no Kaggle.
# libvulkan1 e' obrigatoria: o binario linka libvulkan.so.1 (sem ela: rc=127,
# "error while loading shared libraries: libvulkan.so.1").
subprocess.run("apt-get -qq update && apt-get -qq install -y libvulkan1 libxrender1 libxi6 "
               "libxxf86vm1 libxfixes3 libxkbcommon0 libsm6 libgl1 libegl1 2>/dev/null || true",
               shell=True)

# Sanity check: falha ruidosamente aqui se o Blender ainda nao roda (ex.: lib faltando).
ver = subprocess.run([BLENDER, "--version"], capture_output=True, text=True)
assert ver.returncode == 0, f"Blender nao executa:\n{ver.stderr}"
print(ver.stdout)

## 3. Clonar o repositório
Pule esta célula e ajuste `REPO_DIR` na célula 1 se o repo estiver como *Kaggle Dataset*.

In [ ]:
import os, subprocess

# Clona se ainda nao existe; se JA existe, SINCRONIZA com o remoto (forca hard reset
# p/ pegar as ultimas alteracoes sem risco de conflito). Assim, re-rodar esta celula
# sempre traz a versao mais recente do branch.
if REPO_DIR.startswith("/kaggle/working"):
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", REPO_BRANCH],
                       check=True)
        subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{REPO_BRANCH}"],
                       check=True)
        print("Repo sincronizado com origin/" + REPO_BRANCH)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR],
                       check=True)
        print("Repo clonado")

DG = os.path.join(REPO_DIR, "dataset_generator")
BLENDER_DIR = os.path.join(DG, "blender")
assert os.path.isfile(os.path.join(BLENDER_DIR, "nao_blender.blend")), \
    f"nao_blender.blend nao encontrado em {BLENDER_DIR}"
print("Repo pronto em:", REPO_DIR)

## 4. Instalar `pyyaml` no Python embutido do Blender
O gerador roda dentro do Blender, então a dependência precisa ir no Python **dele**
(não no do Kaggle). `numpy` já vem embutido.

In [ ]:
import glob, subprocess

BPY = glob.glob(os.path.join(BLENDER_HOME, "*", "python", "bin", "python3*"))
BPY = sorted(BPY)[0]
print("Python do Blender:", BPY)

subprocess.run([BPY, "-m", "ensurepip"], check=False)
subprocess.run([BPY, "-m", "pip", "install", "--upgrade", "pip"], check=False)
subprocess.run([BPY, "-m", "pip", "install", "pyyaml"], check=True)
print(subprocess.run([BPY, "-c", "import yaml, numpy; print('yaml', yaml.__version__, '| numpy', numpy.__version__)"],
                     capture_output=True, text=True).stdout)

## 5. Wrapper para habilitar a GPU no Cycles
O `dataset_generator_phobos.py` define o engine como CYCLES mas deixa o *device* em CPU.
Este wrapper é passado com um `--python` **antes** do gerador para ligar OptiX/CUDA.

In [ ]:
GPU_WRAPPER = os.path.join(BLENDER_DIR, "kaggle_enable_gpu.py")

with open(GPU_WRAPPER, "w") as f:
    f.write('''
import bpy

prefs = bpy.context.preferences.addons["cycles"].preferences
chosen = None
for dtype in ("OPTIX", "CUDA"):
    try:
        prefs.compute_device_type = dtype
    except TypeError:
        continue
    prefs.get_devices()
    if any(d.type == dtype for d in prefs.devices):
        chosen = dtype
        break

enabled = 0
if chosen:
    for d in prefs.devices:
        d.use = (d.type == chosen)
        enabled += int(d.use)
    for sc in bpy.data.scenes:
        sc.cycles.device = "GPU"

print(f"[KAGGLE-GPU] compute_device_type={chosen} devices_enabled={enabled}")
''')

print("Wrapper escrito em", GPU_WRAPPER)

## 6. Gerar (N processos por GPU) + monitor
Pega a fatia desta máquina (`shard_range`) e a divide entre `n_gpu × PROCS_PER_GPU`
processos Blender — cada um preso a uma GPU (`CUDA_VISIBLE_DEVICES`), gravando em
`output/g{gpu}_p{k}`. Rodar >1 processo por GPU **sobrepõe o setup de cena (CPU) de
um com o render (GPU) de outro**, subindo o throughput quando a GPU fica ociosa.
`--resume` pula PNGs já feitos (retomável); `--save-every` evita reescrever o COCO
inteiro a cada frame. Um **monitor** mostra a cada 15s as GPUs (`nvidia-smi`) + o
progresso somado; com `NTFY_TOPIC`, publica p/ o painel no seu PC.

In [ ]:
# --- Estado no Drive ANTES de gerar ---
# DYNAMIC: mostra quanto do job ja foi feito (segundo o quadro de claims na nuvem).
#   O restore de cada lote acontece DENTRO do loop, na hora de (re)tomar o lote.
# MANUAL: restaura SO as anotacoes do rank (o --resume pula pelo que ja esta anotado).
import os, sys, subprocess, glob, json
sys.path.insert(0, os.path.join(DG, "scripts"))

if DYNAMIC:
    from coordinator import progress
    done, active, n = progress(BACKUP_REMOTE, TOTAL_SAMPLES, CHUNK)
    print(f"Job {BACKUP_REMOTE}: {done}/{n} lotes concluidos, {active} em andamento "
          f"(~{done*CHUNK}/{TOTAL_SAMPLES} imgs). Esta maquina pega os proximos livres.")
else:
    BACKUP_PATH = f"{BACKUP_REMOTE.rstrip('/')}/rank{RANK}" if BACKUP_REMOTE else None
    if BACKUP_PATH:
        print(f"Restaurando anotacoes de {BACKUP_PATH} -> {OUT_DIR} ...")
        r = subprocess.run(["rclone", "copy", BACKUP_PATH, OUT_DIR,
                            "--include=**/annotations/**", "--transfers=8", "--checkers=8"],
                           capture_output=True, text=True)
        d = 0
        for jf in glob.glob(os.path.join(OUT_DIR, "**", "person_keypoints_*.json"), recursive=True):
            try:
                d += len(json.load(open(jf)).get("images", []))
            except Exception:
                pass
        print(f"rc={r.returncode}  |  {d} amostras ja anotadas (o --resume vai pular essas)")
        if r.returncode != 0:
            print("(ok se for a 1a rodada: ainda nao ha backup no Drive)\n", r.stderr[:200])
    else:
        print("Backup off; nada a restaurar (gera do zero).")

In [ ]:
import subprocess, os, sys, threading, uuid

sys.path.insert(0, os.path.join(DG, "scripts"))
sys.path.insert(0, os.path.join(DG, "src"))
from monitor import watch, flush_backup         # progresso = max(PNGs locais, anotados no JSON)
import coordinator as coord
from nao_coco_pose.sharding import shard_range

# GPUs desta maquina (1 processo Blender por GPU, PROCS_PER_GPU por GPU).
n_gpu = max(1, subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.count("UUID"))
n_proc = n_gpu * PROCS_PER_GPU
os.makedirs(OUT_DIR, exist_ok=True)
print(f"GPUs: {n_gpu}  |  processos: {n_proc} ({PROCS_PER_GPU}/GPU)")


def run_slice(start, num, out_root, backup_remote, target_num):
    """Gera [start, start+num) dividido entre n_proc processos Blender (1 GPU cada),
    com monitor + backup incremental p/ `backup_remote`. Bloqueia ate terminar."""
    os.makedirs(out_root, exist_ok=True)
    base, rem = divmod(num, n_proc)
    shards, cursor, idx = [], start, 0
    for g in range(n_gpu):
        for k in range(PROCS_PER_GPU):
            sn = base + (1 if idx < rem else 0)
            shards.append({"gpu": str(g), "start": cursor, "num": sn,
                           "out": f"{out_root}/g{g}_p{k}"})
            cursor += sn; idx += 1

    procs = []
    for sh in shards:
        if sh["num"] <= 0:
            continue
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=sh["gpu"])
        cmd = [BLENDER, "--background", "nao_blender.blend"]
        if USE_GPU:
            cmd += ["--python", "kaggle_enable_gpu.py"]
        # --resume: pula frames ja feitos (PNG local OU anotacao no JSON, apos o prune).
        cmd += ["--python", "dataset_generator_phobos.py",
                "--", "--num", str(sh["num"]), "--start", str(sh["start"]),
                "--out", sh["out"], "--resume", "--save-every", str(SAVE_EVERY)]
        log = open(sh["out"] + ".log", "w"); sh["logpath"] = log.name
        print(f"  GPU {sh['gpu']} -> {os.path.basename(sh['out'])}: start={sh['start']} num={sh['num']}")
        procs.append((sh, subprocess.Popen(cmd, cwd=BLENDER_DIR, env=env,
                                           stdout=log, stderr=subprocess.STDOUT)))

    stop = threading.Event()
    mon = threading.Thread(target=watch, kwargs=dict(
            out_dir=out_root, num=target_num, interval=15.0, stop=stop,
            ntfy_topic=(NTFY_TOPIC or None),
            backup_remote=backup_remote, backup_every=BACKUP_EVERY), daemon=True)
    mon.start()
    try:
        for sh, p in procs:
            print(f"  {os.path.basename(sh['out'])} terminou  rc={p.wait()}")
    finally:
        stop.set(); mon.join(timeout=17)
        if backup_remote:                         # backup final SINCRONO (ultimo lote + JSON)
            flush_backup(out_root, backup_remote, prune=True)
        for sh, _ in procs:
            with open(sh["logpath"]) as f:
                tail = f.readlines()[-2:]
            print(f"  --- {os.path.basename(sh['out'])} (fim do log) ---\n  {'  '.join(tail)}")


if DYNAMIC:
    # Coordenacao automatica: reivindica o proximo lote livre no Drive, gera, marca
    # como concluido, repete. Sem RANK — quem liga depois pega os proximos lotes.
    MACHINE = os.environ.get("KAGGLE_KERNEL_RUN_ID") or uuid.uuid4().hex[:8]
    print(f"Maquina (id={MACHINE}) entrando no job dinamico de {TOTAL_SAMPLES} (chunk={CHUNK})")
    n_done = 0
    while True:
        claim = coord.claim_next(BACKUP_REMOTE, TOTAL_SAMPLES, CHUNK, MACHINE)
        if claim is None:
            print("[coord] nenhum lote livre — job coberto por esta e outras maquinas.")
            break
        cout = f"{OUT_DIR}/chunk_{claim.idx:05d}"
        crem = f"{BACKUP_REMOTE.rstrip('/')}/chunk_{claim.idx:05d}"
        print(f"\n[coord] lote {claim.idx:05d}: start={claim.start} num={claim.num} -> {crem}")
        # (Re)tomada: se este lote ja tinha progresso na nuvem (retomada apos queda),
        # baixa so os JSON p/ o --resume pular o que ja foi feito.
        subprocess.run(["rclone", "copy", crem, cout, "--include=**/annotations/**",
                        "--transfers=8", "--checkers=8"], capture_output=True)
        # Heartbeat: renova o claim a cada 5min enquanto gera (senao outra maquina
        # poderia considera-lo "morto" e retoma-lo).
        hb_stop = threading.Event()
        def _hb(idx=claim.idx):
            while not hb_stop.wait(300):
                coord.heartbeat(BACKUP_REMOTE, idx, MACHINE)
        hb = threading.Thread(target=_hb, daemon=True); hb.start()
        try:
            run_slice(claim.start, claim.num, cout, crem, claim.num)
        finally:
            hb_stop.set()
        coord.mark_done(BACKUP_REMOTE, claim.idx, MACHINE)
        n_done += 1
        print(f"[coord] lote {claim.idx:05d} concluido ({n_done} nesta maquina).")
    print(f"\nEsta maquina finalizou {n_done} lote(s).")
else:
    # Modo manual: fatia estatica desta maquina (RANK) dentro do job.
    mshard = shard_range(TOTAL_SAMPLES, WORLD_SIZE, RANK)
    bkp = f"{BACKUP_REMOTE.rstrip('/')}/rank{RANK}" if BACKUP_REMOTE else None
    print(f"Maquina rank {RANK}/{WORLD_SIZE}: start={mshard.start} num={mshard.num} -> {bkp}")
    run_slice(mshard.start, mshard.num, OUT_DIR, bkp, mshard.num)

## 7. Validar o JSON COCO e visualizar uma amostra

In [ ]:
import glob, subprocess, json

# Valida o JSON COCO de cada GPU (roda no Python do Kaggle; pycocotools ja disponivel).
# Busca recursiva cobre output/gpu*/annotations/ (1 processo por GPU).
val = os.path.join(DG, "scripts", "validate_dataset.py")
ann = sorted(glob.glob(os.path.join(OUT_DIR, "**", "person_keypoints_*.json"), recursive=True))
env = dict(os.environ, PYTHONPATH=os.path.join(DG, "src"))
for a in ann:
    print("===", os.path.relpath(a, OUT_DIR), "===")
    subprocess.run(["python", val, a], env=env)

In [ ]:
import glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Busca recursiva cobre output/gpu*/images/** (1 processo por GPU).
imgs = sorted(glob.glob(os.path.join(OUT_DIR, "**", "images", "**", "*.png"), recursive=True))
print(f"{len(imgs)} imagens geradas")
if imgs:
    n = min(4, len(imgs))
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    axes = [axes] if n == 1 else axes
    for ax, p in zip(axes, imgs[:n]):
        ax.imshow(mpimg.imread(p)); ax.set_title(os.path.basename(p)); ax.axis("off")
    plt.tight_layout(); plt.show()

import os, shutil

if STORAGE == "tailscale":
    # O dataset ja esta no SEU PC (em ~/synthpose-backup/kaggle). Nao precisa baixar:
    # junte tudo localmente, no PC, apontando --remote pra pasta servida.
    print("Dataset gravado direto no seu PC (pasta servida no `rclone serve webdav`).")
    print("\nNo seu PC, junte tudo com (--remote aceita caminho local):")
    print("  python dataset_generator/scripts/download_and_merge.py \\")
    print("      --remote ~/synthpose-backup/kaggle --out output_merged")
elif BACKUP_REMOTE:
    # Drive: dataset completo esta na nuvem (imagens foram MOVIDAS p/ la). Baixe e junte
    # todas as maquinas/lotes de uma vez, no PC (auto-detecta chunk_*/rank*):
    print("Dataset completo esta no Drive:", BACKUP_REMOTE)
    print("\nNo seu PC (mesmo rclone configurado), baixe e junte tudo com:")
    print(f"  python dataset_generator/scripts/download_and_merge.py \\")
    print(f"      --remote {BACKUP_REMOTE} --out output_merged")
else:
    # Sem backup: dataset so no disco do Kaggle -> zip p/ a aba Output.
    zip_base = "/kaggle/working/nao_dataset"
    zip_path = shutil.make_archive(zip_base, "zip", OUT_DIR)
    print("Pronto:", zip_path, f"({os.path.getsize(zip_path)/1e6:.1f} MB)")
    print("\nBaixe este zip de CADA maquina, depois junte tudo com:")
    print("  python scripts/merge_datasets.py nao_dataset*.zip --out output_merged")

In [ ]:
import os, shutil

if BACKUP_REMOTE:
    # Dataset completo esta no Drive: as imagens foram MOVIDAS p/ la durante a geracao
    # (o zip local estaria quase vazio). Baixe e junte TODAS as maquinas/lotes de uma
    # vez, no SEU PC, com o script dedicado (auto-detecta chunk_*/rank* na nuvem):
    print("Dataset completo esta no Drive:", BACKUP_REMOTE)
    print("\nNo seu PC (com o MESMO rclone configurado), baixe e junte tudo com:")
    print(f"  python dataset_generator/scripts/download_and_merge.py \\")
    print(f"      --remote {BACKUP_REMOTE} --out output_merged")
else:
    # Sem backup: o dataset esta so no disco do Kaggle -> zip p/ a aba Output.
    zip_base = "/kaggle/working/nao_dataset"
    zip_path = shutil.make_archive(zip_base, "zip", OUT_DIR)
    print("Pronto:", zip_path, f"({os.path.getsize(zip_path)/1e6:.1f} MB)")
    print("\nBaixe este zip de CADA maquina, depois junte tudo localmente com:")
    print("  python scripts/merge_datasets.py nao_dataset*.zip --out output_merged")